# Quill — Notebook 08: Phase Oracles & Quantum String Search

In this notebook, we connect a **string predicate** to a quantum operation: a **phase oracle**. We will mark the ASCII character `H` without measuring the register, then use Grover's algorithm to amplify its probability.

> Important: this is a tiny, deliberately bounded demonstration over **one 8-qubit Knot** (256 possible byte values). It is not a practical search engine for arbitrary text. A real multi-character Cord predicate needs a carefully designed reversible circuit.


## 1. The idea: mark a state with its phase

A classical predicate answers yes/no. A quantum phase oracle leaves every basis state alone except the target, whose amplitude changes sign:

$$
O_H|x\rangle =
\begin{cases}
-|H\rangle & x=H,\\
|x\rangle & x\ne H.
\end{cases}
$$

For `H`, ASCII decimal is **72**, binary is **01001000**. The phase change is not directly visible in a single measurement: probabilities depend on squared amplitude magnitudes. Interference with a diffuser makes the phase difference useful.

## 2. Prepare the target Knot and phase-kickback ancilla

We use eight qubits for the Knot and one ancilla. The ancilla is prepared in

$$
| - \rangle = \frac{|0\rangle-|1\rangle}{\sqrt{2}}.
$$

A multi-controlled X flips the ancilla only when the Knot equals `H`. Because the ancilla is $| - \rangle$, that conditional flip contributes a minus sign to the target Knot's amplitude (phase kickback).

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

TARGET = "H"
TARGET_VALUE = ord(TARGET)
N = 8
print(TARGET, TARGET_VALUE, format(TARGET_VALUE, "08b"))


## 3. Build the phase oracle

Qiskit uses qubit 0 as the least-significant bit. So we reverse the displayed binary string when matching target bits to qubit indices. For every target bit that is `0`, apply X before and after the controlled operation; this converts the target pattern into all ones temporarily.


In [ ]:
def apply_phase_oracle(qc, knot, ancilla, target_value):
    target_bits_msb = format(target_value, f"0{len(knot)}b")
    target_bits_lsb_first = target_bits_msb[::-1]
    zero_positions = [i for i, bit in enumerate(target_bits_lsb_first) if bit == "0"]

    for i in zero_positions:
        qc.x(knot[i])
    qc.mcx(list(knot), ancilla[0])
    for i in zero_positions:
        qc.x(knot[i])

# Inspect the oracle circuit on its own.
knot = QuantumRegister(N, "knot")
anc = QuantumRegister(1, "anc")
qc_oracle = QuantumCircuit(knot, anc)
qc_oracle.x(anc[0])
qc_oracle.h(anc[0])  # |->
apply_phase_oracle(qc_oracle, knot, anc, TARGET_VALUE)
qc_oracle.draw("mpl")


## 4. Why the ancilla flips the phase

The controlled-X operation acts as $X$ on the ancilla only for the matching character. But

$$
X| - \rangle = -| - \rangle.
$$

So the matching basis state picks up a minus sign, while all other basis states keep their phase. This is called **phase kickback**.

## 5. Grover diffusion: make the marked character more likely

Start with Hadamard gates on all eight Knot qubits, creating an equal superposition of all 256 byte values. Then repeat the phase oracle and diffuser. For one marked item among $N=256$ possibilities, the useful iteration count is approximately

$$
r \approx \frac{\pi}{4}\sqrt{N} \approx 12.6,
$$

so we will try 12 iterations. Some byte values are not printable characters; this search space is **all 256 byte values**, not only English letters.

In [ ]:
def apply_diffuser(qc, knot):
    # Reflection about the uniform superposition (global phase is irrelevant).
    for q in knot:
        qc.h(q)
        qc.x(q)
    qc.h(knot[-1])
    qc.mcx(list(knot[:-1]), knot[-1])
    qc.h(knot[-1])
    for q in knot:
        qc.x(q)
        qc.h(q)

def build_grover_circuit(target_value=TARGET_VALUE, iterations=12):
    knot = QuantumRegister(N, "knot")
    anc = QuantumRegister(1, "anc")
    out = ClassicalRegister(N, "out")
    qc = QuantumCircuit(knot, anc, out)

    # Uniform superposition across the 256 possible byte values.
    qc.h(knot)

    # Prepare |-> once; it stays |-> throughout the oracle calls.
    qc.x(anc[0])
    qc.h(anc[0])

    for _ in range(iterations):
        apply_phase_oracle(qc, knot, anc, target_value)
        apply_diffuser(qc, knot)

    # Measure only the Knot; the ancilla is not part of the answer.
    qc.measure(knot, out)
    return qc

search_circuit = build_grover_circuit()
search_circuit.draw("mpl", fold=100)


In [ ]:
simulator = AerSimulator()
result = simulator.run(search_circuit, shots=2000).result()
counts = result.get_counts()
plot_histogram(counts, sort="value", number_to_keep=12)


## 6. Check the target result

The count key for `H` is its usual 8-bit binary representation because we measured qubit $i$ into classical bit $i$.

In [ ]:
target_key = format(TARGET_VALUE, "08b")
hits = counts.get(target_key, 0)
print(f"Target: {TARGET!r} (ASCII {TARGET_VALUE}, bits {target_key})")
print(f"Target hits: {hits} / {sum(counts.values())} = {hits / sum(counts.values()):.1%}")
print("Most common byte values:")
for bitstring, count in sorted(counts.items(), key=lambda item: item[1], reverse=True)[:10]:
    value = int(bitstring, 2)
    try:
        char = repr(chr(value))
    except ValueError:
        char = "?"
    print(f"  {bitstring} -> {value:3d} -> {char:>6}: {count}")


## 7. Experiments

1. Change `TARGET` to another ASCII character and rebuild the circuit.
2. Try 0, 1, 6, 12, and 13 Grover iterations. Which gives the strongest target probability?
3. Remove the phase oracle from one iteration. What happens to amplification?
4. Explain why measuring immediately after the phase oracle does not reveal which state was marked.

## Takeaway

A reversible predicate can be used to mark matching basis states with a phase. **Interference**, via Grover diffusion, converts that phase mark into a higher measurement probability. This is the conceptual bridge from Quill's string predicates to quantum search—not a claim that quantum search automatically makes arbitrary text search faster.
